#### Import Packages

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

####Import Cleaned DF

In [ ]:
file_path = "cleaned_df_4lakh.csv"
cleaned_df = pd.read_csv(file_path)
print("flat_df loaded successfully.")
print("First 5 rows:")
display(cleaned_df.tail())

#### Removing all three geospatial parameters

In [ ]:
# Identify fertilizer recommendation columns (Y_cols)
Y_cols = [col for col in cleaned_df.columns if col.startswith(('Maize', 'Rice', 'Wheat'))]

# Identify input feature columns (X_cols)
# Exclude target variables, latitude, longitude, elevation, and categorical location identifiers
columns_to_drop_from_features = ['elevation', 'province', 'district', 'palika'] + Y_cols
X_cols = [col for col in cleaned_df.columns if col not in columns_to_drop_from_features]

# Create X (features) and Y (targets) DataFrames
X = cleaned_df[X_cols]
Y = cleaned_df[Y_cols]

print("Shape of X (features):", X.shape)
print("Shape of Y (targets):", Y.shape)
print("\nFirst 5 rows of X:")
display(X.head())
print("\nFirst 5 rows of Y:")
display(Y.head())

#### Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

models = {
    "Linear Regression": MultiOutputRegressor(
        LinearRegression()
    ),

    "Ridge Regression": MultiOutputRegressor(
        Ridge(alpha=1)
    ),

    "Random Forest": MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=200,
            min_samples_split=5,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        )
    ),

    "XGBoost": MultiOutputRegressor(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=7,
            subsample=0.8,
            random_state=42,
            n_jobs=-1
        )
    ),

    "LightGBM": MultiOutputRegressor(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.1,
            num_leaves=31,
            min_data_in_leaf=20,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
    ),
}

##### Train and evaluate each model ---

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
import warnings
import logging

In [ ]:
warnings.filterwarnings("ignore")

# LightGBM prints warnings through its own logger/stdout.
# This usually suppresses most python-side noise:
logging.getLogger("lightgbm").setLevel(logging.ERROR)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Crop groups
crop_groups = {
    "Maize":  [c for c in Y_cols if c.startswith("Maize")],
    "Rice":   [c for c in Y_cols if c.startswith("Rice")],
    "Wheat":  [c for c in Y_cols if c.startswith("Wheat")]
}

# Fertilizer groups
fert_groups = {}
for c in Y_cols:
    fert = c.split("_")[-1]
    fert_groups.setdefault(fert, []).append(c)

y_idx = {c: i for i, c in enumerate(Y_cols)}

def compute_metrics(y_true, y_pred, cols):
    idx = [y_idx[c] for c in cols]
    yt = y_true.iloc[:, idx]
    yp = y_pred[:, idx]
    return (
        r2_score(yt, yp, multioutput="uniform_average"),
        mean_absolute_error(yt, yp),
        mean_squared_error(yt, yp)  # RMSE
    )


In [ ]:
import time

global_results = {}
crop_results = {}
fert_results = {}

for name, model in models.items():

    print(f"\n🚀 Starting model: {name}")
    start_model = time.time()

    pipe = Pipeline(steps=[("model", model)])

    print("  ▶ Fitting...")
    t0 = time.time()
    pipe.fit(X_train, y_train)
    print(f"  ✅ Fit done in {time.time() - t0:.2f}s")

    print("  ▶ Predicting...")
    t0 = time.time()
    y_pred = pipe.predict(X_test)
    print(f"  ✅ Predict done in {time.time() - t0:.2f}s")

    # ---- Global ----
    print("  ▶ Computing global metrics...")
    global_results[name] = {
        "R²": r2_score(y_test, y_pred, multioutput="uniform_average"),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred)
    }

    # ---- Crop-wise ----
    print("  ▶ Crop-wise metrics...")
    crop_results[name] = {}
    for crop, cols in crop_groups.items():
        r2, _, _ = compute_metrics(y_test, y_pred, cols)
        crop_results[name][crop] = r2

    # ---- Fertilizer-wise ----
    print("  ▶ Fertilizer-wise metrics...")
    fert_results[name] = {}
    for fert, cols in fert_groups.items():
        r2, _, _ = compute_metrics(y_test, y_pred, cols)
        fert_results[name][fert] = r2

    print(f"🏁 Finished {name} in {time.time() - start_model:.2f}s")

In [ ]:
global_df = pd.DataFrame(global_results).T.sort_values("R²", ascending=False)
print("\n=== Global Model Comparison ===")
print(global_df.round(3))